<a href="https://colab.research.google.com/github/gilbertoag2007/fiap-tech-challenge-fase3/blob/main/tech_challenge_fase_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AGRUPANDO AS 5 PARTES DO DATASET

In [1]:
# Clone do repositório no GITHUB
!git clone https://github.com/gilbertoag2007/fiap-tech-challenge-fase3.git

Cloning into 'fiap-tech-challenge-fase3'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 62 (delta 27), reused 6 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 169.84 MiB | 19.87 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [5]:
# ============================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import pandas as pd
import re
import spacy
from pathlib import Path

!pip install -q pandas pyarrow openpyxl
!pip install pandas openpyxl spacy
!python -m spacy download pt_core_news_lg



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 2.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# CARREGAMENTO DOS DADOS

In [3]:
"""
===========================================================================
OBJETIVO
===========================================================================

Este script reúne vários arquivos XLSX que foram divididos devido ao
limite de tamanho do GitHub (ex.: arquivos maiores que 80 MB).

Ao final será gerado um único dataset que servirá de entrada para as
etapas de:

1. Identificação de dados pessoais (PHI)
2. Anonimização
3. Limpeza dos dados
4. Fine-Tuning de uma LLM

Bibliotecas necessárias:

pip install pandas openpyxl

===========================================================================
"""

# ==========================================================================
# Importação das bibliotecas
# ==========================================================================

from pathlib import Path
import pandas as pd

# ==========================================================================
# CONFIGURAÇÕES
# ==========================================================================

# Pasta onde estão os arquivos divididos
PASTA_DADOS = "/content/fiap-tech-challenge-fase3/data"

# Nome do arquivo de saída
ARQUIVO_FINAL = "dataset_medico_completo.xlsx"

# ==========================================================================
# Localiza todos os arquivos XLSX da pasta
# ==========================================================================

print("=" * 70)
print("LOCALIZANDO ARQUIVOS...")
print("=" * 70)

# Procura todos os arquivos .xlsx
arquivos = sorted(Path(PASTA_DADOS).glob("*.xlsx"))

# Verifica se encontrou arquivos
if len(arquivos) == 0:
    raise Exception("Nenhum arquivo XLSX encontrado.")

print(f"Foram encontrados {len(arquivos)} arquivos.\n")

# ==========================================================================
# Leitura dos arquivos
# ==========================================================================

print("=" * 70)
print("LENDO ARQUIVOS...")
print("=" * 70)

lista_dataframes = []

for arquivo in arquivos:

    print(f"Lendo {arquivo.name}")

    # Carrega o arquivo para um DataFrame
    df = pd.read_excel(
        arquivo,
        engine="openpyxl"
    )

    # Guarda o DataFrame em memória
    lista_dataframes.append(df)

# ==========================================================================
# Junta todos os DataFrames
# ==========================================================================

print("\nUnindo arquivos...")

dataset = pd.concat(
    lista_dataframes,
    ignore_index=True
)

print("Arquivos unidos com sucesso!")

# ==========================================================================
# Limpeza básica
# ==========================================================================

print("\nRealizando limpeza inicial...")

# Remove linhas totalmente vazias
dataset.dropna(
    how="all",
    inplace=True
)

# Remove registros duplicados
dataset.drop_duplicates(
    inplace=True
)

# Reinicia a numeração do índice
dataset.reset_index(
    drop=True,
    inplace=True
)

# ==========================================================================
# Informações do dataset
# ==========================================================================

print("\nResumo do dataset")

print("-" * 60)

print(f"Quantidade de registros : {len(dataset):,}")

print(f"Quantidade de colunas   : {len(dataset.columns)}")

print("\nColunas encontradas:\n")

for coluna in dataset.columns:
    print(f" - {coluna}")

# ==========================================================================
# Verificação de valores nulos
# ==========================================================================

print("\nValores ausentes por coluna\n")

print(dataset.isnull().sum())

# ==========================================================================
# Estatísticas básicas
# ==========================================================================

print("\nPrimeiros registros:\n")

print(dataset.head())

# ==========================================================================
# Salva o dataset consolidado
# ==========================================================================

print("\nSalvando arquivo consolidado...")

dataset.to_excel(
    ARQUIVO_FINAL,
    index=False,
    engine="openpyxl"
)

print("\nArquivo salvo com sucesso!")

print(f"\nArquivo gerado: {ARQUIVO_FINAL}")

# ==========================================================================
# Próximas etapas recomendadas
# ==========================================================================

print("\n" + "=" * 70)
print("PRÓXIMAS ETAPAS")
print("=" * 70)

print("""
1) Detectar CPF
2) Detectar RG
3) Detectar CNS
4) Detectar CEP
5) Detectar Telefones
6) Detectar Emails
7) Detectar Datas de nascimento
8) Detectar Endereços
9) Detectar Nomes completos
10) Detectar informações médicas sensíveis

Após essa análise o dataset poderá ser anonimizado antes do Fine-Tuning.
""")

LOCALIZANDO ARQUIVOS...
Foram encontrados 4 arquivos.

LENDO ARQUIVOS...
Lendo dataset_medico_part001.xlsx
Lendo dataset_medico_part002.xlsx
Lendo dataset_medico_part003.xlsx
Lendo dataset_medico_part004.xlsx

Unindo arquivos...
Arquivos unidos com sucesso!

Realizando limpeza inicial...

Resumo do dataset
------------------------------------------------------------
Quantidade de registros : 384,094
Quantidade de colunas   : 6

Colunas encontradas:

 - id
 - pergunta_com_dados_pessoais
 - resposta_formatada
 - condicao
 - especialidade_medica
 - tipo_pergunta

Valores ausentes por coluna

id                             0
pergunta_com_dados_pessoais    0
resposta_formatada             0
condicao                       0
especialidade_medica           0
tipo_pergunta                  0
dtype: int64

Primeiros registros:

       id                        pergunta_com_dados_pessoais  \
0  140527  Uma mãe que ja teve um bebê com gastrosquise p...   
1  425449  Em crises de bronquite, normalm

# DECTECTAR PHI - Protected Health Information

In [4]:
# ============================================================
# IDENTIFICAÇÃO DE DADOS PESSOAIS EM DATASET MÉDICO
# ============================================================
#
# Objetivo:
#   Percorrer todas as colunas de um CSV/XLSX e identificar
#   possíveis informações pessoais ou sensíveis.
#
# Detecta:
#   - CPF
#   - E-mail
#   - Telefone
#   - CEP
#   - Datas
#   - Nomes de pessoas
#   - Organizações
#   - Localidades
#
# IMPORTANTE:
#   Este código NÃO substitui uma ferramenta de anonimização.
#   Ele é uma primeira etapa de análise/curadoria do dataset.
# ============================================================


# ============================================================
# 1. CARREGAR MODELO NLP EM PORTUGUÊS
# ============================================================

# O modelo reconhece entidades como:
#
# PERSON       -> pessoa
# ORG          -> organização
# LOC/GPE      -> localidade
#
# Instalação:
#
# !python -m spacy download pt_core_news_lg

nlp = spacy.load("pt_core_news_lg")


# ============================================================
# 2. EXPRESSÕES REGULARES
# ============================================================

PADROES = {

    # CPF
    "CPF": re.compile(
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b"
    ),

    # E-mail
    "EMAIL": re.compile(
        r"\b[A-Za-z0-9._%+-]+@"
        r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),

    # Telefone brasileiro
    "TELEFONE": re.compile(
        r"(?<!\d)"
        r"(?:\+55\s?)?"
        r"(?:\(?\d{2}\)?\s?)?"
        r"(?:9?\d{4})[-\s]?\d{4}"
        r"(?!\d)"
    ),

    # CEP
    "CEP": re.compile(
        r"\b\d{5}-?\d{3}\b"
    ),

    # Datas no formato dd/mm/yyyy ou dd-mm-yyyy
    "DATA": re.compile(
        r"\b(?:0?[1-9]|[12]\d|3[01])"
        r"[/\-]"
        r"(?:0?[1-9]|1[0-2])"
        r"[/\-]"
        r"(?:19|20)\d{2}\b"
    ),

    # RG genérico
    #
    # Este padrão é propositalmente mais conservador.
    "RG": re.compile(
        r"\b\d{1,2}\.?\d{3}\.?\d{3}-?[0-9Xx]\b"
    )
}


# ============================================================
# 3. FUNÇÃO PARA DETECTAR REGEX
# ============================================================

def detectar_regex(texto):

    encontrados = []

    for tipo, padrao in PADROES.items():

        ocorrencias = padrao.findall(texto)

        if ocorrencias:

            encontrados.append(tipo)

    return encontrados


# ============================================================
# 4. FUNÇÃO PARA DETECTAR ENTIDADES COM spaCy
# ============================================================

def detectar_entidades(texto):

    entidades = []

    doc = nlp(texto)

    for entidade in doc.ents:

        # Pessoa
        if entidade.label_ == "PER":

            entidades.append("NOME_PESSOA")

        # Organização
        elif entidade.label_ == "ORG":

            entidades.append("ORGANIZACAO")

        # Localização
        elif entidade.label_ in ["LOC", "GPE"]:

            entidades.append("LOCALIZACAO")

    return list(set(entidades))


# ============================================================
# 5. ANALISAR UMA CÉLULA
# ============================================================

def analisar_texto(valor):

    if pd.isna(valor):

        return []

    texto = str(valor)

    tipos = []

    # -------------------------------
    # Regex
    # -------------------------------

    tipos.extend(
        detectar_regex(texto)
    )

    # -------------------------------
    # NER
    # -------------------------------

    tipos.extend(
        detectar_entidades(texto)
    )

    # Remover duplicidades
    tipos = sorted(
        list(set(tipos))
    )

    return tipos


# ============================================================
# 6. ANALISAR TODAS AS COLUNAS
# ============================================================

resultados = []


for coluna in df.columns:

    print(f"Analisando coluna: {coluna}")

    total_registros = len(df)

    registros_com_dados = 0

    tipos_encontrados = set()

    exemplos = []

    # --------------------------------------------
    # Percorre os registros da coluna
    # --------------------------------------------

    for indice, valor in df[coluna].items():

        tipos = analisar_texto(valor)

        if tipos:

            registros_com_dados += 1

            tipos_encontrados.update(tipos)

            # Guardamos somente alguns exemplos
            # para não gerar um relatório gigantesco.
            if len(exemplos) < 5:

                exemplos.append({
                    "linha": indice + 2,
                    "valor": str(valor)[:200],
                    "tipos": ", ".join(tipos)
                })

    # --------------------------------------------
    # Percentual de registros suspeitos
    # --------------------------------------------

    percentual = 0

    if total_registros > 0:

        percentual = (
            registros_com_dados /
            total_registros
        ) * 100

    resultados.append({

        "coluna": coluna,

        "total_registros": total_registros,

        "registros_com_dados_pessoais":
            registros_com_dados,

        "percentual_suspeito":
            round(percentual, 2),

        "tipos_detectados":
            ", ".join(sorted(tipos_encontrados)),

        "exemplos":
            str(exemplos)

    })


# ============================================================
# 7. CRIAR DATAFRAME DE RESULTADO
# ============================================================

relatorio = pd.DataFrame(resultados)


# ============================================================
# 8. EXIBIR RESULTADO
# ============================================================

print("\n")
print("=" * 80)
print("RELATÓRIO DE DADOS PESSOAIS")
print("=" * 80)

print(relatorio.to_string(index=False))


# ============================================================
# 9. SALVAR RELATÓRIO
# ============================================================

# Arquivo onde será salvo o relatório
ARQUIVO_RELATORIO = "relatorio_dados_pessoais.xlsx"

relatorio.to_excel(
    ARQUIVO_RELATORIO,
    index=False
)

print("\nRelatório salvo em:")
print(ARQUIVO_RELATORIO)

NameError: name 'spacy' is not defined